In [4]:
# Cell 1: Install Dependencies
# !pip install kornia

In [5]:
# Cell 2: Imports & Configuration
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
import math
import scipy.sparse.linalg
from typing import Union, Tuple

import torch
import torch.nn as nn
import torch.fft
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
from torchvision.io.image import decode_jpeg, encode_jpeg

try:
    import kornia
except ImportError:
    raise ImportError("Kornia is required. Please run '!pip install kornia' first.")

# --- CONFIGURATION ---
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("Using CPU")
    DEVICE = "cpu"

# EfficientNet-B5 Native Resolution
IMAGE_SIZE = 456 

# BATCH SIZE STRATEGY (P100 - 16GB VRAM)
# P100 has 16GB, same as a single T4. 
# B5 at 456x456 is heavy. We stick to 4 to be safe.
BATCH_SIZE = 4          
ACCUMULATION_STEPS = 8  # 4 * 8 = 32 Effective Batch Size
EPOCHS = 4
LR = 1e-4
VAL_SPLIT = 0.1

# --- DATA PATHS ---
TRAIN_SHARD_PATHS = [
    "/kaggle/input/datasets/zeus005/shard-0/shard_0",
    "/kaggle/input/datasets/zeus005/shard-1/shard_1",
    "/kaggle/input/datasets/zeus005/shard-2/shard_2"
]
VAL_IMAGES_PATH = "/kaggle/input/datasets/zeus005/validation-set-ivp/val_images"

Using GPU: Tesla P100-PCIE-16GB


In [ ]:
# Cell 3: Distortion Library (Original & Complete)

# ==========================================
# CONSOLIDATED DISTORTION LIBRARY (ORIGINAL)
# ==========================================

# --- PART 1: UTILS & MATH ---
def sign(x: float) -> int: return 1 if x >= 0 else -1

def mapmm(x: torch.Tensor) -> torch.Tensor:
    minx, maxx = torch.min(x), torch.max(x)
    return (x - minx) / (maxx - minx) if minx < maxx else x

def fspecial(filter_type: str, p2: Union[int, Tuple[int, int]], p3: Union[int, float] = None) -> np.ndarray:
    if filter_type == 'gaussian':
        m, n = [(ss - 1.) / 2. for ss in p2]
        y, x = np.ogrid[-m:m + 1, -n:n + 1]
        h = np.exp(-(x * x + y * y) / (2. * p3 * p3))
        h[h < np.finfo(h.dtype).eps * h.max()] = 0
        sumh = h.sum()
        if sumh != 0: h /= sumh
        return h
    elif filter_type == 'disk':
        rad = p2; crad = math.ceil(rad - 0.5)
        x, y = np.ogrid[-rad: rad + 1, -rad: rad + 1]
        y = np.tile(y.transpose(), y.shape[1]); x = np.tile(x, x.shape[0]).transpose()
        y = np.abs(y); x = np.abs(x)
        maxxy = np.maximum(x, y); minxy = np.minimum(x, y)
        r1 = (rad ** 2 - (maxxy + 0.5) ** 2); r2 = (rad ** 2 - (minxy - 0.5) ** 2)
        warn_m1 = r1 ** 0.5 if (r1 > 0).all() else 0
        warn_m2 = r2 ** 0.5 if (r2 > 0).all() else 0
        m1 = (rad ** 2 < (maxxy + 0.5) ** 2 + (minxy - 0.5) ** 2) * (minxy - 0.5) + \
             (rad ** 2 >= (maxxy + 0.5) ** 2 + (minxy - 0.5) ** 2) * warn_m1
        m2 = (rad ** 2 > (maxxy - 0.5) ** 2 + (minxy + 0.5) ** 2) * (minxy + 0.5) + \
             (rad ** 2 <= (maxxy - 0.5) ** 2 + (minxy + 0.5) ** 2) * warn_m2
        sgrid = (rad ** 2 * (0.5 * (np.arcsin(m2 / rad) - np.arcsin(m1 / rad)) +
                             0.25 * (np.sin(2 * np.arcsin(m2 / rad)) - np.sin(2 * np.arcsin(m1 / rad)))) - (
                         maxxy - 0.5) * (m2 - m1) + (m1 - minxy + 0.5)) * np.logical_or(
            np.logical_and((rad ** 2 < (maxxy + 0.5) ** 2 + (minxy + 0.5) ** 2),
                           (rad ** 2 > (maxxy - 0.5) ** 2 + (minxy - 0.5) ** 2)),
            np.logical_and(np.logical_and(minxy == 0, maxxy - 0.5 < rad), maxxy + 0.5 >= rad))
        sgrid = sgrid + ((maxxy + 0.5) ** 2 + (minxy + 0.5) ** 2 < rad ** 2)
        sgrid[crad, crad] = np.minimum(math.pi * rad ** 2, math.pi / 2)
        if (crad > 0) and (rad > crad - 0.5) and (rad ** 2 < (crad - 0.5) ** 2 + 0.25):
            m1 = np.sqrt(rad ** 2 - (crad - 0.5) ** 2); m1n = m1 / rad
            sg0 = 2 * (rad ** 2 * (0.5 * np.arcsin(m1n) + 0.25 * np.sin(2 * np.arcsin(m1n))) - m1 * (crad - 0.5))
            sgrid[2 * crad, crad] = sg0; sgrid[crad, 2 * crad] = sg0
            sgrid[crad, 0] = sg0; sgrid[0, crad] = sg0
            sgrid[2 * crad, crad] -= sg0; sgrid[crad, 2 * crad] -= sg0
            sgrid[crad, 2] -= sg0; sgrid[2, crad] = sgrid[2, crad + 1] - sg0
        sgrid[crad, crad] = np.minimum(sgrid[crad, crad], 1)
        h = sgrid / np.sum(sgrid)
        return h
    else: raise NotImplementedError(f"Filter type {filter_type} not implemented")

def filter2D(img: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    img = img.float()
    k1 = kernel.size(-2); k2 = kernel.size(-1)
    b, c, h, w = img.size()
    if k1 % 2 == 1 or k2 % 2 == 1:
        img = F.pad(img, (k2 // 2, k2 // 2, k1 // 2, k1 // 2), mode='replicate')
    else: raise ValueError('Wrong kernel size')
    ph, pw = img.size()[-2:]
    if kernel.size(0) == 1:
        img = img.view(b * c, 1, ph, pw)
        kernel = kernel.view(1, 1, k1, k2)
        return F.conv2d(img, kernel, padding=0).view(b, c, h, w)
    else:
        img = img.view(1, b * c, ph, pw)
        kernel = kernel.view(b, 1, k1, k2).repeat(1, c, 1, 1).view(b * c, 1, k1, k2)
        return F.conv2d(img, kernel, groups=b * c).view(b, c, h, w)

def spline(x: np.ndarray, y: np.ndarray) -> tuple:
    n = x.shape[0]; dd = 1; dx = np.diff(x); divdif = np.diff(y) / dx
    if n == 3:
        y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]
        dlk = y[[2, 1, 0]].shape[0]; l = x[[0, 2]].shape[0] - 1; dl = np.prod(dd) * l
        k = np.fix(dlk / dl + 100 * 2.2204e-16); pp = (x[[0, 2]], y[[2, 1, 0]], l, int(k), dd)
    elif n > 3:
        b = np.zeros(n); b[1:n - 1] = 3 * (dx[1:n] * divdif[0:n - 2] + dx[0:n - 2] * divdif[1:n])
        x31 = x[2] - x[0]; xn = x[n - 1] - x[n - 3]
        b[0] = ((dx[0] + 2 * x31) * dx[1] * divdif[0] + dx[0] ** 2 * divdif[1]) / x31
        b[n - 1] = (dx[n - 2] ** 2 * divdif[n - 3] + (2 * xn + dx[n - 2]) * dx[n - 3] * divdif[n - 2]) / xn
        dxt = dx.T; c = np.zeros((3, 5))
        c[0, :] = [x31] + list(dxt[0:n - 2]) + [0]
        c[1, :] = [dxt[1]] + list(2 * (dxt[1:n - 1] + dxt[0:n - 2])) + [dxt[n - 3]]
        c[2, :] = [0] + list(dxt[1:n - 1]) + [xn]
        c = scipy.sparse.dia_matrix((c, [-1, 0, 1]), shape=(5, 5))
        c = scipy.sparse.csc_matrix(c); ic = scipy.sparse.linalg.inv(c); s = b * ic
        dxd = dx; dzzdx = (divdif - s[0:n - 1]) / dxd; dzdxdx = (s[1:n] - divdif) / dxd
        coefs = np.vstack(((dzdxdx - dzzdx) / dxd, 2 * dzzdx - dzdxdx, s[0:n - 1], y[0:n - 1])).T
        pp = (x, coefs, x.shape[0], x.shape[0], 1)
    else: raise ValueError('x.shape[0] must be >= 3')
    return pp

def histc(x: torch.Tensor, binranges: torch.Tensor) -> torch.Tensor:
    indices = torch.bucketize(x, binranges)
    return torch.remainder(indices, len(binranges)) - 1

def ppval(pp: tuple, xx: torch.Tensor) -> torch.Tensor:
    lx = torch.numel(xx); xs = xx.reshape(1, lx)
    b, c, l, k, dd = pp
    b = torch.as_tensor(b, device=xx.device)
    ranges = b.clone(); ranges[0] = -torch.inf; ranges[-1] = torch.inf
    index = histc(xs, ranges); xs = xs - b[index]
    c = torch.as_tensor(c, device=xx.device)
    if len(c.shape) == 1:
        v = c[0]
        for i in range(1, k): v = xs * v + c[i]
    else:
        v = c[index, 0]
        for i in range(1, k - 1): v = xs * v + c[index, i]
    v = v.view(xx.shape)
    return v

def curves(xx: torch.Tensor, coef: float) -> torch.Tensor:
    if isinstance(coef, list): coef = [[0.3, 0.5, 0.7], [coef[0], 0.5, coef[1]]]
    else: coef = [[0.5], [coef]]
    x = np.array([0] + [p for p in coef[0]] + [1])
    y = np.array([0] + [p for p in coef[1]] + [1])
    cs = spline(x, y); yy = ppval(cs, xx); yy = torch.clamp(yy, 0, 1)
    return yy

def bilinear_interpolate_torch(im: torch.Tensor, x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    dtype_long = torch.LongTensor
    x0 = torch.floor(x).type(dtype_long).to(im.device); x1 = x0 + 1
    y0 = torch.floor(y).type(dtype_long).to(im.device); y1 = y0 + 1
    x0 = torch.clamp(x0, 0, im.shape[1] - 1); x1 = torch.clamp(x1, 0, im.shape[1] - 1)
    y0 = torch.clamp(y0, 0, im.shape[0] - 1); y1 = torch.clamp(y1, 0, im.shape[0] - 1)
    Ia = im[y0, x0]; Ib = im[y1, x0]; Ic = im[y0, x1]; Id = im[y1, x1]
    R1 = Ia * (x1 - x) / (x1 - x0 + eps) + Ic * (x - x0) / (x1 - x0 + eps)
    R2 = Ib * (x1 - x) / (x1 - x0 + eps) + Id * (x - x0) / (x1 - x0 + eps)
    P = R1 * (y1 - y) / (y1 - y0 + eps) + R2 * (y - y0) / (y1 - y0 + eps)
    return P

def imscatter(x: torch.Tensor, amount: float, iterations=1) -> torch.Tensor:
    y = x
    for i in range(iterations):
        shiftmap = torch.randn((2, x.shape[1], x.shape[2]), device=x.device) * amount
        sy = shiftmap[0, :, :]; sx = shiftmap[1, :, :]
        m_sx = torch.ceil(torch.abs(torch.max(sx))).to(torch.int32)
        m_sy = torch.ceil(torch.abs(torch.max(sy))).to(torch.int32)
        y = F.pad(y, (m_sy, m_sy), mode='replicate')
        y = F.pad(y.transpose(2, 1), (m_sx, m_sx), mode='replicate').transpose(2, 1)
        sy = F.pad(sy, (m_sy, m_sy), mode='replicate')
        sy = F.pad(sy.transpose(1, 0), (m_sx, m_sx), mode='replicate').transpose(1, 0)
        sx = F.pad(sx, (m_sy, m_sy), mode='replicate')
        sx = F.pad(sx.transpose(1, 0), (m_sx, m_sx), mode='replicate').transpose(1, 0)
        xx, yy = torch.as_tensor(np.mgrid[0:y.shape[1], 0:y.shape[2]], device=x.device)
        z = torch.zeros_like(y)
        bx = (xx - sx); by = (yy - sy)
        for i in range(3):
            j = bilinear_interpolate_torch(y[i, ...], by, bx)
            z[i, :, :] = j
        y = z[:, m_sy:m_sy + x.shape[1], m_sx:m_sx + x.shape[2]]
    return y

# --- PART 2: DISTORTION FUNCTIONS ---
def gaussian_blur(x: torch.Tensor, blur_sigma: int = 0.1) -> torch.Tensor:
    fs = 2 * math.ceil(2 * blur_sigma) + 1
    h = fspecial('gaussian', (fs, fs), blur_sigma)
    h = torch.from_numpy(h).float()
    if len(x.shape) == 3: x = x.unsqueeze(0)
    y = filter2D(x, h.unsqueeze(0)).squeeze(0)
    return y

def lens_blur(x: torch.Tensor, radius: int) -> torch.Tensor:
    h = fspecial('disk', radius)
    h = torch.from_numpy(h).float()
    if len(x.shape) == 3: x = x.unsqueeze(0)
    y = filter2D(x, h.unsqueeze(0)).squeeze(0)
    return y

def color_saturation(x: torch.Tensor, factor: int) -> torch.Tensor:
    x = x[[2, 1, 0], ...]
    hsv = kornia.color.rgb_to_hsv(x)
    hsv[1, ...] *= factor
    y = kornia.color.hsv_to_rgb(hsv)
    return y[[2, 1, 0], ...]

def color_shift(x: torch.Tensor, amount: int) -> torch.Tensor:
    def perc(x, perc):
        xs = torch.sort(x)
        i = len(xs) * perc / 100.
        i = max(min(i, len(xs)), 1)
        v = xs[round(i - 1)]
        return v
    gray = kornia.color.rgb_to_grayscale(x)
    gradxy = kornia.filters.spatial_gradient(gray.unsqueeze(0), 'diff')
    e = torch.sum(gradxy ** 2, 2) ** 0.5
    fs = 2 * math.ceil(2 * 4) + 1
    h = fspecial('gaussian', (fs, fs), 4)
    h = torch.from_numpy(h).float()
    e = filter2D(e, h.unsqueeze(0))
    mine, maxe = torch.min(e), torch.max(e)
    if mine < maxe: e = (e - mine) / (maxe - mine)
    percdev = [1, 1]
    valuehi = perc(e, 100 - percdev[1]); valuelo = 1 - perc(1 - e, 100 - percdev[0])
    e = torch.max(torch.min(e, valuehi), valuelo)
    channel = 1
    g = x[channel, :, :]
    a = np.random.random((1, 2))
    amount_shift = np.round(a / (np.sum(a ** 2) ** 0.5) * amount)[0].astype(int)
    y = F.pad(g, (amount_shift[0], amount_shift[0]), mode='replicate')
    y = F.pad(y.transpose(1, 0), (amount_shift[1], amount_shift[1]), mode='replicate').transpose(1, 0)
    y = torch.roll(y, (amount_shift[0], amount_shift[1]), dims=(0, 1))
    if amount_shift[1] != 0: y = y[amount_shift[1]:-amount_shift[1], ...]
    if amount_shift[0] != 0: y = y[..., amount_shift[0]:-amount_shift[0]]
    yblend = y * e + x[channel, ...] * (1 - e)
    x[channel, ...] = yblend
    return x

def jpeg(x: torch.Tensor, quality: int) -> torch.Tensor:
    x_loc = x.clone() * 255.
    x_loc = x_loc.clamp(0, 255)
    y = encode_jpeg(x_loc.byte().cpu(), quality=quality)
    y = (decode_jpeg(y) / 255.).to(torch.float32)
    return y

def white_noise(x: torch.Tensor, var: float, clip: bool = True, rounds: bool = False) -> torch.Tensor:
    noise = torch.randn(*x.size(), dtype=x.dtype) * math.sqrt(var)
    y = x + noise
    if clip and rounds: y = torch.clip((y * 255.0).round(), 0, 255) / 255.
    elif clip: y = torch.clip(y, 0, 1)
    elif rounds: y = (y * 255.0).round() / 255.
    return y

def impulse_noise(x: torch.Tensor, d: float, s_vs_p: float = 0.5) -> torch.Tensor:
    num_sp = int(d * x.shape[0] * x.shape[1] * x.shape[2])
    coords = np.concatenate((np.random.randint(0, 3, (num_sp, 1)),
                             np.random.randint(0, x.shape[1], (num_sp, 1)),
                             np.random.randint(0, x.shape[2], (num_sp, 1))), 1)
    num_salt = int(s_vs_p * num_sp)
    coords_salt = coords[:num_salt].transpose(1, 0)
    coords_pepper = coords[num_salt:].transpose(1, 0)
    x[tuple(coords_salt)] = 1; x[tuple(coords_pepper)] = 0
    return x

def brighten(x: torch.Tensor, amount: float) -> torch.Tensor:
    x = x[[2, 1, 0]]
    lab = kornia.color.rgb_to_lab(x)
    l = lab[0, ...] / 100.
    l_ = curves(l, 0.5 + amount / 2)
    lab[0, ...] = l_ * 100.
    y = curves(x, 0.5 + amount / 2)
    j = torch.clamp(kornia.color.lab_to_rgb(lab), 0, 1)
    y = (2 * y + j) / 3
    return y[[2, 1, 0]]

def darken(x: torch.Tensor, amount: float, dolab: bool = False) -> torch.Tensor:
    x = x[[2, 1, 0], :, :]
    lab = kornia.color.rgb_to_lab(x)
    if dolab:
        l = lab[0, ...] / 100.
        l_ = curves(l, 0.5 + amount / 2)
        lab[0, ...] = l_ * 100.
    y = curves(x, 0.5 - amount / 2)
    if dolab:
        j = torch.clamp(kornia.color.lab_to_rgb(lab), 0, 1)
        y = (2 * y + j) / 3
    return y[[2, 1, 0]]

def jitter(x: torch.Tensor, amount: float) -> torch.Tensor:
    y = imscatter(x, amount, 5)
    return y

def quantization(x: torch.Tensor, levels: int) -> torch.Tensor:
    image = kornia.color.rgb_to_grayscale(x) * 255
    image = image.cpu().numpy()
    num_classes = levels
    hist, bins = np.histogram(image, num_classes, [0, 255])
    return_thresholds = np.zeros(num_classes - 1)
    for i in range(num_classes - 1): return_thresholds[i] = bins[i + 1]
    bins = torch.tensor([0] + return_thresholds.tolist() + [256])
    bins = bins.type(torch.int)
    image = torch.bucketize(x.contiguous() * 255., bins).to(torch.float32)
    image = mapmm(image)
    return image

def linear_contrast_change(x: torch.Tensor, amount: float) -> torch.Tensor:
    y = curves(x, [0.25 - amount / 4, 0.75 + amount / 4])
    return y

# --- PART 3: MAIN CALL ---
distortion_groups = {
    "blur": ["gausblur", "lensblur"],
    "color_distortion": ["colorshift", "colorsat"],
    "jpeg": ["jpeg"],
    "noise": ["whitenoise", "impulsenoise"],
    "brightness_change": ["brighten", "darken"],
    "spatial_distortion": ["jitter",  "quantization"],
    "sharpness_contrast": ["lincontrchange"],
}

distortion_range = {
    "gausblur": [0.1, 0.5, 1, 2, 5],
    "lensblur": [1, 2, 4, 6, 8],
    "colorshift": [1, 3, 6, 8, 12],
    "colorsat": [0.4, 0.2, 0.1, 0, -0.4],
    "jpeg": [43, 36, 24, 7, 4],
    "whitenoise": [0.001, 0.002, 0.003, 0.005, 0.01],
    "impulsenoise": [0.001, 0.005, 0.01, 0.02, 0.03],
    "brighten": [0.1, 0.2, 0.4, 0.7, 1.1],
    "darken": [0.05, 0.1, 0.2, 0.4, 0.8],
    "jitter": [0.05, 0.1, 0.2, 0.5, 1],
    "quantization": [20, 16, 13, 10, 7],
    "lincontrchange": [0., 0.15, -0.4, 0.3, -0.6],
}

distortion_functions = {
    "gausblur": gaussian_blur, "lensblur": lens_blur,
    "colorshift": color_shift, "colorsat": color_saturation,
    "jpeg": jpeg, "whitenoise": white_noise, "impulsenoise": impulse_noise,
    "brighten": brighten, "darken": darken,
    "jitter": jitter, "quantization": quantization,
    "lincontrchange": linear_contrast_change
}

def get_distortions_composition(max_distortions: int = 3, num_levels: int = 5):
    MEAN = 0
    STD = 2.5
    num_distortions = random.randint(1, max_distortions)
    groups = random.sample(list(distortion_groups.keys()), num_distortions)
    distortions = [random.choice(distortion_groups[group]) for group in groups]
    distort_functions = [distortion_functions[dist] for dist in distortions]
    probabilities = [1 / (STD * np.sqrt(2 * np.pi)) * np.exp(-((i - MEAN) ** 2) / (2 * STD ** 2)) for i in range(num_levels)]
    normalized_probabilities = [prob / sum(probabilities) for prob in probabilities]
    distort_values = [np.random.choice(distortion_range[dist][:num_levels], p=normalized_probabilities) for dist in distortions]
    return distort_functions, distort_values

def distort_images(image: torch.Tensor, distort_functions: list = None, distort_values: list = None,
                   max_distortions: int = 3, num_levels: int = 5) -> torch.Tensor:
    image_loc = image.clone()
    if distort_functions is None or distort_values is None:
        distort_functions, distort_values = get_distortions_composition(max_distortions, num_levels)
    for distortion, value in zip(distort_functions, distort_values):
        image_loc = distortion(image_loc, value)
        image_loc = image_loc.to(torch.float32)
        image_loc = torch.clip(image_loc, 0, 1)
    return image_loc, distort_functions, distort_values

print("✅ Distortion Library Loaded (ALL Ops Enabled).")

✅ Distortion Library Loaded (ALL Ops Enabled).


In [7]:
# Cell 4: Dataset Class (With 60% Distortion Prob)
class AIGenDetDataset(Dataset):
    def __init__(self, shard_paths, mode='train', distortion_prob=0.6):
        self.mode = mode
        self.distortion_prob = distortion_prob # 60% chance to distort
        self.shard_dirs = [x for x in shard_paths if os.path.exists(x)]
        
        if not self.shard_dirs:
            print(f"Warning: Checked {shard_paths}, found none.")
            self.label_df = pd.DataFrame()
        else:
            label_dfs = []
            for s_dir in self.shard_dirs:
                csv_path = os.path.join(s_dir, 'labels.csv')
                if os.path.exists(csv_path):
                    df = pd.read_csv(csv_path, index_col=0)
                    df['shard_path'] = s_dir 
                    label_dfs.append(df)
            self.label_df = pd.concat(label_dfs, ignore_index=True) if label_dfs else pd.DataFrame()
            if 'image_name' not in self.label_df.columns:
                 self.label_df['image_name'] = self.label_df.index
            print(f'Dataset Initialized. Loaded {len(self.label_df)} images.')

        # Base resize transform
        self.resize = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.label_df)

    def __getitem__(self, idx):
        img_path = os.path.join(self.label_df.loc[idx, 'shard_path'], 'images', str(self.label_df.loc[idx, 'image_name']))
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))

        # 1. Resize & Convert
        image = self.resize(image)
        img_tensor = self.to_tensor(image)

        # 2. PROBABILISTIC DISTORTIONS
        # Only run the heavy 'distort_images' function 60% of the time.
        if self.mode == 'train':
            if random.random() < self.distortion_prob:
                try:
                    img_tensor, _, _ = distort_images(img_tensor)
                except:
                    pass

        # 3. Normalize
        img_tensor = self.normalize(img_tensor)

        # 4. Return RGB Only (Model handles FFT)
        label = torch.tensor(self.label_df.loc[idx, 'label'], dtype=torch.float32)
        return img_tensor, label

In [8]:
# Cell 5: Model (GPU-Accelerated FFT + SE Intact)
class EffNetWithFFT(nn.Module):
    def __init__(self):
        super().__init__()
        # Load B5 (SE blocks intact)
        self.backbone = efficientnet_b5(weights=EfficientNet_B5_Weights.DEFAULT)
        
        # Modify first layer for 4 channels
        original_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=4, 
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=False
        )
        
        # Initialize weights
        with torch.no_grad():
            new_conv.weight[:, :3, :, :] = original_conv.weight
            new_conv.weight[:, 3:4, :, :] = torch.mean(original_conv.weight, dim=1, keepdim=True)
            
        self.backbone.features[0][0] = new_conv
        
        # Modify classifier
        self.backbone.classifier[1] = nn.Linear(self.backbone.classifier[1].in_features, 1)

    def get_fft_feature_gpu(self, x):
        # x is [Batch, 3, H, W] on GPU
        # Convert to Grayscale
        gray = 0.299 * x[:, 0, :, :] + 0.587 * x[:, 1, :, :] + 0.114 * x[:, 2, :, :]
        gray = gray.unsqueeze(1) # [Batch, 1, H, W]
        
        # FFT on GPU
        fft = torch.fft.fft2(gray)
        fft_shift = torch.fft.fftshift(fft)
        mag = torch.abs(fft_shift)
        spectrum = torch.log(mag + 1e-6)
        
        # Vectorized Normalization (Min-Max per image)
        flat = spectrum.view(spectrum.size(0), -1)
        min_val = flat.min(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
        max_val = flat.max(dim=1, keepdim=True)[0].view(-1, 1, 1, 1)
        
        spectrum = (spectrum - min_val) / (max_val - min_val + 1e-6)
        return spectrum

    def forward(self, x):
        # 1. Calculate FFT (Happens fast on GPU)
        fft_map = self.get_fft_feature_gpu(x)
        
        # 2. Stack (3 channel RGB + 1 channel FFT = 4 channels)
        x_combined = torch.cat([x, fft_map], dim=1)
        
        # 3. Pass to Backbone
        return self.backbone(x_combined)

def get_model():
    print("Loading EfficientNet-B5 (GPU-Accelerated FFT, SE Intact)...")
    model = EffNetWithFFT()
    return model

In [9]:
# Cell 6: Training Loop
def evaluate(model, loader):
    model.eval()
    all_labels, all_probs = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(DEVICE)
            # forward() handles FFT automatically
            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    preds = (np.array(all_probs) >= 0.5).astype(int)
    acc = (preds == np.array(all_labels)).mean()
    print(f"   >>> Val Accuracy: {acc:.4f}")
    return acc

def train_model():
    # 1. Load Data
    full_dataset = AIGenDetDataset(TRAIN_SHARD_PATHS, mode='train', distortion_prob=0.4)
    val_size = int(len(full_dataset) * VAL_SPLIT)
    train_size = len(full_dataset) - val_size
    train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, 
                              num_workers=2, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, 
                            num_workers=2)

    # 2. Setup Model
    model = get_model()
    model.to(DEVICE)
    
    # NOTE: P100 is usually single GPU, so DataParallel is removed.
    # If you somehow have 2x P100s, uncomment the next two lines:
    # if torch.cuda.device_count() > 1:
    #     model = nn.DataParallel(model)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler() 
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, 
        steps_per_epoch=len(train_loader) // ACCUMULATION_STEPS, 
        epochs=EPOCHS
    )

    print(f"Starting Training on {DEVICE}...")
    
    for epoch in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for i, (images, labels) in enumerate(pbar):
            images = images.to(DEVICE)
            labels = labels.unsqueeze(1).to(DEVICE)
            
            # Mixed Precision Context
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss = loss / ACCUMULATION_STEPS
            
            scaler.scale(loss).backward()

            if (i + 1) % ACCUMULATION_STEPS == 0:
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
                
                current_loss = loss.item() * ACCUMULATION_STEPS
                running_loss += current_loss
                pbar.set_postfix({"Loss": f"{current_loss:.4f}"})

        # Save Checkpoint
        torch.save(model.state_dict(), f"effnet_b5_optimized_epoch_{epoch+1}.pth")
        
        evaluate(model, val_loader)

    return model

In [10]:
# Cell 7: Execute Training
if __name__ == "__main__":
    trained_model = train_model()

Dataset Initialized. Loaded 150000 images.
Loading EfficientNet-B5 (GPU-Accelerated FFT, SE Intact)...
Downloading: "https://download.pytorch.org/models/efficientnet_b5_lukemelas-1a07897c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b5_lukemelas-1a07897c.pth


100%|██████████| 117M/117M [00:00<00:00, 230MB/s] 


Starting Training on cuda...


/tmp/ipykernel_55/675938462.py:43: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Epoch 1/4:   0%|          | 0/33750 [00:00<?, ?it/s]

...

Epoch 4/4:   0%|          | 0/33750 [00:00<?, ?it/s]

/tmp/ipykernel_55/723464387.py:76: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]
/tmp/ipykernel_55/723464387.py:76: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]


Validating:   0%|          | 0/3750 [00:00<?, ?it/s]

/tmp/ipykernel_55/723464387.py:76: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]
/tmp/ipykernel_55/723464387.py:76: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  y[1:3] = divdif; y[2] = np.diff(divdif.T).T / (x[2] - x[0]); y[1] -= y[2] * dx[0]


   >>> Val Accuracy: 0.9571


In [11]:
# Cell 8: Inference
def run_inference(model_state_path=None, val_dir=VAL_IMAGES_PATH):
    print(f"Running Inference on: {val_dir}")
    
    # 1. Setup Transforms
    resize_transform = transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))
    to_tensor = transforms.ToTensor()
    normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    
    # 2. Load Model
    model = get_model() # Get architecture
    if model_state_path:
        model.load_state_dict(torch.load(model_state_path))
    elif 'trained_model' in globals():
        model = trained_model
    else:
        print("No model found. Please provide a path or run training.")
        return

    model.to(DEVICE)
    model.eval()
    
    # 3. Gather Images
    val_path = Path(val_dir)
    image_files = sorted([f for f in val_path.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.jpeg']])
    
    if len(image_files) == 0:
        print("No images found! Check path.")
        return

    predictions = []
    
    with torch.no_grad():
        for img_file in tqdm(image_files, desc="Inferencing"):
            try:
                # Load
                img = Image.open(img_file).convert("RGB")
                img = resize_transform(img)
                img_tensor = to_tensor(img)
                img_tensor = normalize(img_tensor)
                
                # Batch Dimension [1, 3, H, W]
                img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
                
                # Predict (Model handles FFT)
                logits = model(img_tensor)
                prob = torch.sigmoid(logits).item()
                
                predictions.append({"image_name": img_file.name, "score": prob})
            except Exception as e:
                print(f"Error {img_file.name}: {e}")

    # 4. Save
    df = pd.DataFrame(predictions)
    df.to_csv("submission.csv", index=False)
    print(f"Saved submission.csv with {len(df)} rows.")

run_inference()

Running Inference on: /kaggle/input/datasets/zeus005/validation-set-ivp/val_images
Loading EfficientNet-B5 (GPU-Accelerated FFT, SE Intact)...


Inferencing:   0%|          | 0/10000 [00:00<?, ?it/s]

Saved submission.csv with 10000 rows.
